<a href="https://colab.research.google.com/github/LoneWolf2026/Neural-Network-for-Nuclear-Binding-Energy-Prediction-/blob/main/Nuclear_Binding_Energy_NeuralNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#1. split data between training, testing, and validation
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [15]:
data_df_2020 = pd.read_csv('/content/drive/MyDrive/AME_Dataset_2020.csv',header =None)
data_df_2020 = data_df_2020.drop([0, 1]).reset_index(drop=True) #removes neutron and proton from first two rows
#data_df_2020 = data_df_2020.iloc[:,[0,1,2,3,5,6,9,10,7,8,4]].reset_index(drop=True) #Rearrange data so Binding Energy and Elements are the right most columns (column indexes are not altered)
data_df_2020.head()

,0,1,2,3,4,5,6,7,8,9,10
0,0,1,1,2,H,13135.722895,0.000015,1112.28310,0.00020,14101.777844,0.000015
1,1,2,1,3,H,14949.810900,0.000080,2827.26540,0.00030,16049.281320,0.000080
2,-1,1,2,3,He,14931.218880,0.000060,2572.68044,0.00015,16029.321970,0.000060
3,-3,0,3,3,Li,28667.000000,2000.000000,-2267.00000,667.00000,30775.000000,2147.000000
4,2,3,1,4,H,24621.129000,100.000000,1720.44910,25.00000,26431.867000,107.354000


In [39]:
og_data = data_df_2020.copy()

In [59]:
#Find out what to do with the error calculations listed for each data point

input = np.array(data_df_2020.iloc[:,[0,1,2,3,5,6,9,10]].values)
output = np.array(data_df_2020.iloc[:,[7]].values) #We start with Binding Energy, then we will decide if element name and uncertainty can be incorporated later

#print(input)
#print(output)

In [60]:
input_train,input_test,output_train,output_test = train_test_split(input,output,test_size = 0.3)
input_test,input_val,output_test,output_val = train_test_split(input_test,output_test,test_size = 0.5)

In [61]:
class dataset(Dataset):
  def __init__(self,input,output):
    self.input = torch.tensor(input, dtype = torch.float32).to(device)
    self.output = torch.tensor(output, dtype = torch.float32).to(device)

  def __len__(self):
    return len(self.input)

  def __getitem__(self,index):
    return self.input[index], self.output[index]

In [62]:
training_data = dataset(input_train,output_train)
validation_data = dataset(input_val,output_val)
testing_data = dataset(input_test,output_test)

In [63]:
train_dataloader = DataLoader(training_data,batch_size = 4,shuffle = True)
val_dataloader = DataLoader(validation_data,batch_size = 4,shuffle = True)
test_dataloader = DataLoader(testing_data,batch_size = 4,shuffle = True)

In [64]:
for i,b in train_dataloader:
  print(i)
  print("===========")
  print(b)
  break

tensor([[ 3.0000e+01,  1.1900e+02,  8.9000e+01,  2.0800e+02,  1.0761e+04,
          6.4483e+01,  1.1552e+04,  6.9225e+01],
        [ 1.9000e+01,  6.9000e+01,  5.0000e+01,  1.1900e+02, -9.0065e+04,
          7.2500e-01,  9.0331e+05,  7.7800e-01],
        [ 2.2000e+01,  7.9000e+01,  5.7000e+01,  1.3600e+02, -8.6037e+04,
          5.3171e+01,  9.0763e+05,  5.7081e+01],
        [ 5.0000e+00,  6.0000e+01,  5.5000e+01,  1.1500e+02, -5.9699e+04,
          1.0200e+02,  9.3591e+05,  1.1000e+02]], device='cuda:0')
tensor([[7684.8291],
        [8499.4492],
        [8376.0508],
        [8216.0000]], device='cuda:0')


In [65]:
Hidden_Neurons = 8
class BindingE_NeuralNet(nn.Module):
  def __init__(self):
    super(BindingE_NeuralNet,self).__init__()

    self.layer1 = nn.Linear(input_train.shape[1],Hidden_Neurons)
    self.OutLayer = nn.Linear(Hidden_Neurons,1) #We want at least one output: Binding Energy (uncertainty could be added later)
    self.Relu = nn.ReLU()

  def forward(self,X):

    X = self.layer1(X)
    X = self.OutLayer(X)
    X = self.Relu(X)
    #Return to this Model and refine architecture

    return X


BE_NeuralNet = BindingE_NeuralNet().to('cuda')

In [66]:
summary(BE_NeuralNet,(input.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                    [-1, 8]              72
            Linear-2                    [-1, 1]               9
              ReLU-3                    [-1, 1]               0
Total params: 81
Trainable params: 81
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [67]:
criterion = nn.BCELoss()
optimizer = Adam(BE_NeuralNet.parameters(), lr = 1e-3)

In [68]:
for data in train_dataloader:
  inputs, labels = data
  print(inputs)
  print(labels)
  break

tensor([[ 1.8000e+01,  3.8000e+01,  2.0000e+01,  5.8000e+01, -1.5300e+03,
          5.0000e+02,  9.9836e+05,  5.3700e+02],
        [-5.0000e+00,  1.0000e+01,  1.5000e+01,  2.5000e+01,  2.0190e+04,
          4.0000e+02,  2.1675e+04,  4.2900e+02],
        [ 2.0000e+00,  2.9000e+01,  2.7000e+01,  5.6000e+01, -5.6041e+04,
          4.7500e-01,  9.3984e+05,  5.1000e-01],
        [ 4.4000e+01,  1.3900e+02,  9.5000e+01,  2.3400e+02,  4.4462e+04,
          1.6000e+02,  4.7731e+04,  1.7200e+02]], device='cuda:0')
tensor([[7828.0000],
        [6794.0000],
        [8694.8389],
        [7564.0000]], device='cuda:0')


In [69]:
total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

epochs = 10
for epoch in range(epochs):
  total_acc_train = 0
  total_loss_train = 0
  total_acc_validation = 0
  total_loss_validation = 0

  for data in train_dataloader:
    inputs, labels = data

    prediction = BE_NeuralNet(inputs)

    batch_loss = criterion(prediction,labels)
    total_loss_train += batch_loss.item()

    acc = ((prediction).round() == labels).sum().item()

    total_acc_train += acc

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for data in val_dataloader:
      inputs, labels = data

      prediction = BE_NeuralNet(inputs)
      batch_loss = criterion(prediction,labels)

      total_loss_validation += batch_loss.item()
      acc = ((prediction).round() == labels).sum().item()

      total_acc_validation += acc

  total_loss_train_plot.append(round(total_loss_train/1000,4))
  total_loss_validation_plot.append(round(total_loss_validation/1000,4))

  total_acc_train_plot.append(round(total_acc_train/training_data.__len__()*100,4))
  total_acc_validation_plot.append(round(total_acc_validation/validation_data.__len__()*100,4))

  print(f'''Epoch: {epoch+1} Train_Loss: {round(total_loss_train/1000,4)} Train_Accuracy: {round(total_acc_train/training_data.__len__()*100,4)}
        Validation Loss: {round(total_loss_validation/1000,4)} Validation_Accuracy: {round(total_acc_validation/validation_data.__len__()*100,4)}''')
  print("=="*25)

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
